# 03 · Clinical baselines and XGBoost — SepsisWatch Z

1. **SIRS and qSOFA baselines** computed from vital signs and labs. They represent current practice: this is what we compare against.
2. **XGBoost** on the 140 features from `02_features`, trained on hospital A with early stopping on validation.
3. **Feature importance** with `pred_contribs=True` (evidence `19`) and a per-alert explanation (the 3 features pushing risk up the most) for the Granite notes.
4. Saves `xgb.json` (for the `scorer`) and `preds_xgb.pkl` (for `05_eval`).

> The train/val/test split uses exactly the same code as `04_nnpa_mlp`, so the ensemble compares the same rows.

## 1. Environment

In [ ]:
import os, json, time, platform, warnings, gc
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score, average_precision_score
import matplotlib.pyplot as plt

print(f"Machine: {platform.machine()} | xgboost {xgb.__version__} | pandas {pd.__version__} | CPUs: {os.cpu_count()}")

## 2. Configuration

In [ ]:
DATA_DIR    = os.path.expanduser("~/shared/sepsis/data/processed")
RAW_ALL     = f"{DATA_DIR}/sepsis_all.pkl"
FEAT_PATH   = f"{DATA_DIR}/features.pkl"
FLIST_PATH  = f"{DATA_DIR}/feature_list.json"
MODEL_DIR   = os.path.expanduser("~/shared/sepsis/models")    # outside the repo
REPO        = os.path.abspath("..")                            # notebooks run from notebooks/
FIG_DIR     = f"{REPO}/docs/figures"

LABEL_COL   = "SepsisLabel"
DROP_FEATS  = []          # keep identical in 03 and 04, e.g. ["ICULOS"]
SEED        = 42

NTHREAD     = 3           # the VM has 3 IFL vCPUs
XGB_PARAMS = {
    "objective": "binary:logistic", "eval_metric": ["auc", "aucpr"], "tree_method": "hist",
    "max_depth": 6, "eta": 0.05, "subsample": 0.8, "colsample_bytree": 0.8,
    "min_child_weight": 5, "lambda": 1.0, "nthread": NTHREAD, "seed": SEED,
}
NUM_ROUNDS, EARLY_STOP = 1500, 50

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

## 3. Load features and split (identical to `04`)

In [ ]:
t0 = time.perf_counter()
df = pd.read_pickle(FEAT_PATH)
with open(FLIST_PATH) as f:
    flist = json.load(f)
FEATURES = flist if isinstance(flist, list) else (flist.get("features") or flist.get("feature_list") or next(iter(flist.values())))
FEATURES = [c for c in FEATURES if c not in DROP_FEATS]
missing = [c for c in FEATURES + [LABEL_COL, "patient_id", "hour"] if c not in df.columns]
assert not missing, f"Columns not found: {missing[:10]}"
print(f"features.pkl: {df.shape} loaded in {time.perf_counter()-t0:.1f}s | {df.memory_usage(deep=True).sum()/1e9:.2f} GB")
print(f"Features used: {len(FEATURES)} | other columns: {[c for c in df.columns if c not in FEATURES][:12]}")

SPLIT_COL = next((c for c in ("split", "set", "fold", "subset") if c in df.columns), None)
if SPLIT_COL:
    split = df[SPLIT_COL].astype(str).str.lower().replace({"validation": "val", "valid": "val"})
    print(f"Using the existing split (column '{SPLIT_COL}')")
else:
    warnings.warn("No split column found: rebuilding it (same code in notebooks 03 and 04).")
    split = pd.Series("test", index=df.index)
    a = df["hospital"].astype(str) == "A"
    per_patient = df.loc[a].groupby("patient_id")[LABEL_COL].max()
    rng = np.random.default_rng(SEED)
    val_ids = set()
    for lab in (0, 1):
        ids = per_patient.index[per_patient.values == lab].to_numpy()
        rng.shuffle(ids)
        val_ids.update(ids[: int(round(len(ids) * 0.2))])
    split[a] = np.where(df.loc[a, "patient_id"].isin(val_ids), "val", "train")

idx = {s: np.flatnonzero(split.to_numpy() == s) for s in ("train", "val", "test")}
y = df[LABEL_COL].to_numpy().astype(np.float32)
for s, ii in idx.items():
    print(f"{s:>5}: {len(ii):>9,} rows | {df['patient_id'].iloc[ii].nunique():>6,} patients | label=1: {y[ii].mean():.2%}")
assert all(len(v) for v in idx.values()), "A split is empty"
assert not set(df["patient_id"].iloc[idx["train"]]) & set(df["patient_id"].iloc[idx["test"]]), "Patients shared by train and test"

## 4. Clinical baselines: SIRS and qSOFA
`02_features` already computes both scores from the carried-forward vitals and labs (columns `SIRS` and `qSOFA`), so they are reused here. If they are missing, they are recomputed from `sepsis_all.pkl`.

- **SIRS (0–4):** heart rate > 90; temperature > 38 or < 36 °C; respiratory rate > 20 or PaCO₂ < 32; WBC > 12 or < 4 (×10³/µL). Classic alert: SIRS ≥ 2.
- **qSOFA (0–2 with this data):** respiratory rate ≥ 22 and systolic BP ≤ 100. Mental status is not in the dataset. Alert: qSOFA ≥ 2.

In [ ]:
META = df[[c for c in ("patient_id", "hospital", "hour", "Age", "Gender", "ICULOS") if c in df.columns]].copy()
META["patient_id"] = META["patient_id"].astype(str)        # plain strings: safe merges in 05
META["split"] = split.to_numpy()
META[LABEL_COL] = y

if {"SIRS", "qSOFA"} <= set(df.columns):
    META["sirs"] = df["SIRS"].fillna(0).to_numpy().astype(np.int8)
    META["qsofa"] = df["qSOFA"].fillna(0).to_numpy().astype(np.int8)
    print("SIRS and qSOFA taken from features.pkl")
else:
    VARS = ["HR", "Temp", "Resp", "PaCO2", "WBC", "SBP"]
    raw = pd.read_pickle(RAW_ALL)[["patient_id", "hour"] + VARS].sort_values(["patient_id", "hour"])
    raw[VARS] = raw.groupby("patient_id", sort=False, observed=True)[VARS].ffill()
    raw["sirs"] = ((raw.HR > 90).astype(np.int8) + ((raw.Temp > 38) | (raw.Temp < 36)).astype(np.int8)
                   + ((raw.Resp > 20) | (raw.PaCO2 < 32)).astype(np.int8) + ((raw.WBC > 12) | (raw.WBC < 4)).astype(np.int8))
    raw["qsofa"] = (raw.Resp >= 22).astype(np.int8) + (raw.SBP <= 100).astype(np.int8)
    raw["patient_id"] = raw["patient_id"].astype(str)
    META = META.merge(raw[["patient_id", "hour", "sirs", "qsofa"]], on=["patient_id", "hour"], how="left", validate="1:1")
    assert len(META) == len(df), "The merge changed the number of rows"
    META[["sirs", "qsofa"]] = META[["sirs", "qsofa"]].fillna(0).astype(np.int8)
    del raw; gc.collect()
    print("SIRS and qSOFA recomputed from sepsis_all.pkl")

print(META.groupby(LABEL_COL)[["sirs", "qsofa"]].mean().rename(index={0.0: "label 0", 1.0: "label 1"}).round(2))

## 5. Train XGBoost (CPU, z17 SIMD vector units)
XGBoost handles `NaN` natively: no imputation. `scale_pos_weight` offsets the class imbalance like the network's `pos_weight`, so both probabilities live on similar scales for the ensemble.

In [ ]:
X = df[FEATURES].to_numpy(dtype=np.float32, copy=True)
del df; gc.collect()
print(f"X: {X.shape} {X.dtype} | {X.nbytes/1e9:.2f} GB")

n_pos = y[idx["train"]].sum(); n_neg = len(idx["train"]) - n_pos
params = dict(XGB_PARAMS, scale_pos_weight=float(n_neg / n_pos))

dtrain = xgb.QuantileDMatrix(X[idx["train"]], label=y[idx["train"]], feature_names=FEATURES)
dval   = xgb.QuantileDMatrix(X[idx["val"]],   label=y[idx["val"]],   feature_names=FEATURES, ref=dtrain)

t0 = time.perf_counter()
booster = xgb.train(params, dtrain, num_boost_round=NUM_ROUNDS, evals=[(dtrain, "train"), (dval, "val")],
                    early_stopping_rounds=EARLY_STOP, verbose_eval=50)
train_s = time.perf_counter() - t0
BEST = booster.best_iteration
print(f"\nBest iteration: {BEST} | val AUPRC {booster.best_score:.4f} | training time {train_s:.0f}s")
del dtrain; gc.collect()

## 6. Predictions and hourly metrics (validation and hospital B)

In [ ]:
def predict_xgb(rows, chunk=200_000):
    out = []
    for i in range(0, len(rows), chunk):
        d = xgb.DMatrix(X[rows[i:i+chunk]], feature_names=FEATURES, nthread=NTHREAD)
        out.append(booster.predict(d, iteration_range=(0, BEST + 1)))
    return np.concatenate(out).astype(np.float32)

p_xgb, pred_s = {}, {}
for s in ("val", "test"):
    t0 = time.perf_counter(); p_xgb[s] = predict_xgb(idx[s]); pred_s[s] = time.perf_counter() - t0

rows = []
for s in ("val", "test"):
    ys, m = y[idx[s]], META.iloc[idx[s]]
    for name, score in (("SIRS", m.sirs.to_numpy()), ("qSOFA", m.qsofa.to_numpy()), ("XGBoost", p_xgb[s])):
        rows.append({"split": s, "model": name, "AUROC": roc_auc_score(ys, score),
                     "AUPRC": average_precision_score(ys, score), "prevalence": ys.mean()})
metrics = pd.DataFrame(rows)
display(metrics.style.format({"AUROC": "{:.4f}", "AUPRC": "{:.4f}", "prevalence": "{:.4f}"}))
print(f"Scoring {len(idx['test']):,} patient-hours from hospital B with XGBoost: {pred_s['test']:.1f}s")

## 7. Which features drive the model? (evidence 19)
XGBoost's native SHAP contributions (`pred_contribs=True`) on a sample of hospital B: how much each feature pushes the risk of each hour.

In [ ]:
rng = np.random.default_rng(SEED)
sample = rng.choice(idx["test"], size=min(50_000, len(idx["test"])), replace=False)
contrib = booster.predict(xgb.DMatrix(X[sample], feature_names=FEATURES, nthread=NTHREAD),
                          pred_contribs=True, iteration_range=(0, BEST + 1))[:, :-1]   # last column = bias
importance = pd.Series(np.abs(contrib).mean(0), index=FEATURES).sort_values(ascending=False)

top = importance.head(20)[::-1]
fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(top.index, top.values, color="#0f62fe")
ax.set_xlabel("mean |SHAP| contribution to risk (log-odds)")
ax.set_title("SepsisWatch Z · XGBoost · top 20 features (hospital B)")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/xgb_importance.png", dpi=150); plt.show()

rank = {c: i + 1 for i, c in enumerate(importance.index)}
if "ICULOS" in rank:
    print(f"ICULOS ranks {rank['ICULOS']} of {len(importance)}."
          + ("  ⚠️ It is in the top 5: check whether it is a shortcut (hours in ICU) and consider DROP_FEATS=['ICULOS']." if rank["ICULOS"] <= 5 else ""))

## 8. Explaining one alert (input for Granite)
For each Critical patient, the 3 features that raise the risk the most, with their values. This is what Granite receives (no identifiers).

In [ ]:
def top3(row_x):
    c = booster.predict(xgb.DMatrix(row_x[None, :], feature_names=FEATURES), pred_contribs=True,
                        iteration_range=(0, BEST + 1))[0, :-1]
    order = np.argsort(-c)[:3]
    return [{"feature": FEATURES[j], "value": None if np.isnan(row_x[j]) else round(float(row_x[j]), 2),
             "contribution": round(float(c[j]), 3)} for j in order]

i_max = idx["test"][int(np.argmax(p_xgb["test"]))]
m = META.iloc[i_max]
print(f"Highest-risk hour in hospital B: risk {p_xgb['test'].max():.3f} | age {m.get('Age', 'NA')} | ICULOS {m.get('ICULOS', 'NA')}")
for t in top3(X[i_max]):
    print(f"  {t['feature']:<28} value {t['value']}  → +{t['contribution']}")

## 9. Save model and predictions

In [ ]:
booster.save_model(f"{MODEL_DIR}/xgb.json")
with open(f"{MODEL_DIR}/xgb_config.json", "w") as f:
    json.dump({"features": FEATURES, "best_iteration": int(BEST), "params": params, "train_s": train_s,
               "metrics": metrics.to_dict(orient="records"), "top20": importance.head(20).round(5).to_dict()},
              f, indent=2, default=float)

pred = META.iloc[np.concatenate([idx["val"], idx["test"]])].reset_index(drop=True)
pred["p_xgb"] = np.concatenate([p_xgb["val"], p_xgb["test"]])
pred.to_pickle(f"{MODEL_DIR}/preds_xgb.pkl")

print("Saved to", MODEL_DIR)
for fn in sorted(os.listdir(MODEL_DIR)):
    print(f"  {fn:<28} {os.path.getsize(f'{MODEL_DIR}/{fn}')/1e6:8.2f} MB")
print("preds_xgb.pkl columns:", list(pred.columns))

**For the `scorer`** — load XGBoost and score one hour (same 140 features, no imputation):

```python
import json, numpy as np, xgboost as xgb
cfg = json.load(open(".../models/xgb_config.json"))
bst = xgb.Booster(); bst.load_model(".../models/xgb.json")
x = np.array([[row.get(c, np.nan) for c in cfg["features"]]], dtype=np.float32)
d = xgb.DMatrix(x, feature_names=cfg["features"])
p = float(bst.predict(d, iteration_range=(0, cfg["best_iteration"] + 1))[0])
contrib = bst.predict(d, pred_contribs=True, iteration_range=(0, cfg["best_iteration"] + 1))[0, :-1]  # top-3 for Granite
```

Tip for the `replayer`: instead of recomputing the 140 features online, replay the precomputed hospital B rows from `features.pkl`, hour by hour. That is exactly what the `scorer` would see in production, and it avoids rewriting the `02_features` code.